# Synapse Parameterization Example

Demonstrates the block-based synapse parameterization workflow.

This example:
1. Connects to the staging entitycore database
2. Selects a synaptome (circuit of scale "single")
3. Defines distributions, synaptic models, neuron sets, and model assigners as composable blocks
4. Assembles a `SynapseParameterizationSingleConfig`
5. Runs the `SynapseParameterizationTask` to add physiological parameters to the synaptome

## Setup: Imports and database client

In [ ]:
import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token

In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)

## Select a synaptome

A synaptome is a circuit of scale `"single"` (one real neuron with all its synapses).
Replace the ID below with a valid synaptome from the staging database.

In [ ]:
synaptome_id = "<PASTE-SYNAPTOME-ID-HERE>"

synaptome_from_id = obi.MEModelWithSynapsesCircuitFromID(id_str=synaptome_id)

## Define distributions

Distributions specify how physiological parameter values are drawn.
Each distribution is stored in the config's `distributions` dictionary and
referenced by synaptic models via `obi.AllDistributionsReference`.

In [ ]:
# Excitatory pathway distributions
u_hill_exc = obi.PositiveFloatConstantDistribution(value=0.5)
gsyn_exc = obi.PositiveFloatUniformDistribution(low=0.5, high=2.0, random_seed=42)

# Inhibitory pathway distributions
u_hill_inh = obi.PositiveFloatConstantDistribution(value=0.25)
gsyn_inh = obi.PositiveFloatUniformDistribution(low=1.0, high=4.0, random_seed=42)

distributions = {
    "u_hill_exc": u_hill_exc,
    "gsyn_exc": gsyn_exc,
    "u_hill_inh": u_hill_inh,
    "gsyn_inh": gsyn_inh,
}

## Define synaptic models

Synaptic models define which distributions to use for each physiological parameter.
The Tsodyks-Markram model parameterizes `u_hill_coefficient` and `gsyn`.

In [ ]:
def dist_ref(name: str) -> obi.AllDistributionsReference:
    """Helper to create a distribution reference."""
    return obi.AllDistributionsReference(
        block_dict_name="distributions", block_name=name
    )


# Excitatory synaptic model
tm_exc = obi.TsodyksMarkramSynapseParameterization(
    u_hill_coefficient_distribution=dist_ref("u_hill_exc"),
    gsyn_distribution=dist_ref("gsyn_exc"),
    u_hill_coefficient_shared_within=False,
    gsyn_distribution_shared_within=False,
)

# Inhibitory synaptic model (with correlation between u_hill and gsyn)
tm_inh = obi.CorrelatedTsodyksMarkramSynapseParameterization(
    u_hill_coefficient_distribution=dist_ref("u_hill_inh"),
    gsyn_distribution=dist_ref("gsyn_inh"),
    u_hill_coefficient_shared_within=False,
    gsyn_distribution_shared_within=False,
    u_hill_coefficient_and_gsyn_correlation=-0.3,
)

synaptic_models = {
    "tm_exc": tm_exc,
    "tm_inh": tm_inh,
}

## Define neuron sets

Neuron sets identify source and target neuron populations for synapse model assignment.

In [ ]:
all_neurons = obi.AllNeurons()

neuron_sets = {
    "all": all_neurons,
}

## Define synapse model assigners

Each assigner maps a synaptic model to a source-target neuron set pair.
This controls which synapses get parameterized with which model.

In [ ]:
def neuron_ref(name: str) -> obi.NeuronSetReference:
    """Helper to create a neuron set reference."""
    return obi.NeuronSetReference(block_dict_name="neuron_sets", block_name=name)


def model_ref(name: str) -> obi.SynapticModelReference:
    """Helper to create a synaptic model reference."""
    return obi.SynapticModelReference(block_dict_name="synaptic_models", block_name=name)


# Assign excitatory TM model to all-to-all synapses
assigner_exc = obi.InterNeuronSetSynapticModelAssigner(
    source_neuron_set=neuron_ref("all"),
    targeted_neuron_set=neuron_ref("all"),
    synaptic_model=model_ref("tm_exc"),
    overwrite_if_exists=False,
    random_seed=123456,
)

synapse_model_assigners = {
    "exc_all_to_all": assigner_exc,
}

## Assemble configuration

In [ ]:
info = obi.Info(
    campaign_name="Synapse Parameterization Example",
    campaign_description="Parameterize synaptome with Tsodyks-Markram model distributions",
)

initialize = obi.SynapseParameterizationSingleConfig.Initialize(
    synaptome=synaptome_from_id,
)

task_config = obi.SynapseParameterizationSingleConfig(
    info=info,
    initialize=initialize,
    distributions=distributions,
    synaptic_models=synaptic_models,
    neuron_sets=neuron_sets,
    synapse_model_assigners=synapse_model_assigners,
    coordinate_output_root="synaptome_output",
)

## Run parameterization task

In [ ]:
task = obi.SynapseParameterizationTask(config=task_config)

In [ ]:
result = task.execute(db_client=db_client, entity_cache=True)

## Inspect the parameterized synaptome

In [ ]:
synaptome = task._synaptome.sonata_circuit
for epop in synaptome.edges.population_names:
    edges = synaptome.edges[epop]
    print(f"{epop}: {edges.size} synapses")
    props = [p for p in edges.property_names if p not in ("@source_node", "@target_node")]
    syn_table = edges.afferent_edges(node_id=0, properties=["@source_node"] + props)
    print(syn_table)
    syn_table[props].hist(figsize=(10, 3))